In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# pip install gensim
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(400)

In [ ]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
df = pd.read_csv("/content/2021_Sep5_97k_hydrated_tweets.csv")

In [ ]:
df.drop(columns=['id_str',"in_reply_to_screen_name","is_retweet"], inplace=True)

In [ ]:
# Function to Clean the Tweet.

import re
def clean_tweet(tweet):
    return ' '.join(re.sub('(\\\\n)|(b\"[^0-9A-Za-z A-Za-z0-9 \t]+)|(b\'[^0-9A-Za-z]+)|(b\"[A-Za-z0-9]+)|(b\'[A-Za-z0-9]+)|(b\'#[A-Za-z0-9]+)|(b\'@[A-Za-z0-9]+)|(\\\\x[A-Za-z0-9]+)|(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|([RT])', ' ', str(tweet).lower()).split())


In [ ]:
# Call function to get Clean tweets
df["CleanTweet"] = df['text'].apply(lambda x : clean_tweet(x))
df.tail()

,favorite_count,source,text,created_at,retweet_count,CleanTweet
78210,0,PostBeyond,b'An online tool used by people in Italy to ma...,Mon Sep 06 04:02:23 +0000 2021,2,online tool used by people in italy to manage ...
78211,1,Khoros CX,"b""@seany_boy71 Hi there, it sounds like you're...",Mon Sep 06 04:02:28 +0000 2021,0,seany boy71 hi there it sounds like you re tal...
78212,0,Twitter Web App,b'@TracyBethHoeg REALLY? YOU THINK? GIVEN THER...,Mon Sep 06 04:02:23 +0000 2021,0,tracybethhoeg really you think given there is ...
78213,0,Twitter for iPhone,b'So in an area thats of \xe2\x80\x98major out...,Mon Sep 06 04:02:22 +0000 2021,0,in an area thats of outbreak concern they put ...
78214,0,Twitter for Android,b'@GovAbbott and State Republicans allowed my ...,Mon Sep 06 04:02:16 +0000 2021,0,govabbott and state republicans allowed my you...


In [ ]:
df.drop(df.index[df.CleanTweet.eq("")], inplace=True)

In [ ]:
df.head()

,favorite_count,source,text,created_at,retweet_count,CleanTweet
0,1,Twitter for Android,"b""Don't believe it. https://t.co/VFfhPArn1a""",Sat Sep 03 04:09:04 +0000 2022,0,t believe it
1,0,cmssocialservice,b'Surat records 11 new Covid-19 cases https://...,Sat Sep 03 04:06:03 +0000 2022,0,records 11 new covid 19 cases
2,38,TweetDeck,b'Hennepin County will no longer require COVID...,Sat Sep 03 04:09:00 +0000 2022,6,county will no longer require covid 19 vaccine...
3,2,Twitter Web App,"b'As of Sept 1, the capital has given over 63....",Sat Sep 03 04:08:52 +0000 2022,0,of sept 1 the capital has given over 63 2 mill...
4,1,Twitter for Android,b'@KushThrough @BlazedRTs @rtsmallstreams @Rts...,Sat Sep 03 04:15:46 +0000 2022,3,kushthrough ww nowplaying x lilroyce her ev


In [ ]:
stemmer = SnowballStemmer(language='english')
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

# Tokenize and lemmatize
def preprocess(text):
    result=[]
    for token in gensim.utils.simple_preprocess(text) :
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
            
    return result

In [ ]:
complete_text = ' '.join(df["CleanTweet"])

In [ ]:
data = df.CleanTweet.values.tolist()

In [ ]:
pprint(data[:2])

['t believe it', 'records 11 new covid 19 cases']


In [ ]:
complete_text


't believe it records 11 new covid 19 cases county will no longer require covid 19 vaccines for its employees of sept 1 the capital has given over 63 2 million doses of covid19 vaccines to over 23 6 million people kushthrough ww nowplaying x lilroyce her ev can t be good for you themixmedic nowplaying x lilroyce her evil spine feat makrazy total of 174 new cases of covid 19 were reported during the last 24 hours across the state odisha odishanews appointed by the israeli moh to investigate covid 19 vaccine side effects warned the ministry it could b covid 19 closed our campuses down the idea that every student has a computer a word processing program sta thetruthsucks12 why get poked in 2021 when covid 19 has a 98 survival rate got a bridge in brooklyn for sale yo federal government is fucking trash dubai reins in hospitality as covid 19 cases rise is hilarious you can make this up the camp counselor who tried to overturn a decisive democratic election pminmangaluru pm modi mentioned t

In [ ]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
processed_docs = preprocess(complete_text)

In [ ]:
dictionary = gensim.corpora.Dictionary([processed_docs])

In [ ]:
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 aaaaaaaaa
1 aajtak
2 aapi
3 aapl
4 aarogya
5 aaron
6 aarondodd
7 aaronla
8 aaronparna
9 aarospeir
10 aayeff


In [ ]:
#dictionary.filter_extremes(no_below=20, no_above=0.1, keep_n= 1000000)
#dictionary.filter_extremes(no_above=0.70)

In [ ]:
dictionary

In [ ]:
#bow_corpus = dictionary.doc2bow(processed_docs)

bow_corpus = [dictionary.doc2bow(processed_docs),]
#bow_corpus = [dictionary.doc2bow(text) for text in processed_docs]
print(bow_corpus[:1])

[[]]


In [ ]:
#LDA
lda_model = gensim.models.ldamodel.LdaModel(corpus=bow_corpus, 
                                            num_topics = 20, 
                                            id2word = dictionary,
                                            random_state=100,
                                            update_every=1,
                                            chunksize=100,
                                            alpha='auto',
                                            per_word_topics=True,
                                            passes = 50)

In [ ]:
for idx, topic in lda_model.print_topics(-1):
    print("Topic: {} \nWords: {}".format(idx, topic ))
    print("\n")

In [ ]:
import re
import numpy as np
import pandas as  pd
from pprint import pprint# Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
#from gensim.models import CoherenceModel# spaCy for preprocessing
import spacy# Plotting tools
#import pyLDAvis
#import pyLDAvis.gensim
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
!pip3 install spacy

!python3 -m spacy download en #Language model

#pip3 install gensim # For topic modeling

#pip3 install pyLDAvis # For visualizing topic models

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
2022-10-06 08:48:23.784403: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 12.8 MB 2.8 MB/s 
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
# NLTK Stop words
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

In [ ]:
def sent_to_words(sentences):
  for sentence in sentences:
    yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))            #deacc=True removes punctuations

In [ ]:
data_words = list(sent_to_words(data))
print(data_words[:1])

[['police', 'and', 'firefighters', 'sue', 'governor', 'over', 'covid', 'vaccine', 'mandate']]


In [ ]:
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)
print(trigram_mod[bigram_mod[data_words[0]]])


/usr/local/lib/python3.7/dist-packages/gensim/models/phrases.py:598: UserWarning: For a faster implementation, use the gensim.models.phrases.Phraser class
  warnings.warn("For a faster implementation, use the gensim.models.phrases.Phraser class")


['police', 'and', 'firefighters_sue_governor', 'over', 'covid', 'vaccine', 'mandate']


In [ ]:
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [ ]:
data_words_nostops = remove_stopwords(data_words)

data_words_bigrams = make_bigrams(data_words_nostops)

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [ ]:
print(data_lemmatized[:5])

[['police', 'firefighters_sue', 'governor', 'mandate'], ['new', 'case', 'report', 'record', 'covid', 'positive', 'case'], ['sm', 'supermall', 'easy', 'step'], ['hear', 'man', 'get'], ['report', 'new', 'case', 'last_hour']]


In [ ]:
id2word = corpora.Dictionary(data_lemmatized)  
texts = data_lemmatized  
corpus = [id2word.doc2bow(text) for text in texts]  
print(corpus[:1])

[[(0, 1), (1, 1), (2, 1), (3, 1)]]


In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=20, 
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=50,
                                           alpha='auto',
                                           per_word_topics=True)

In [ ]:
from pprint import pprint
pprint(lda_model.print_topics())

[(0,
  '0.096*"vaccinate" + 0.061*"think" + 0.036*"last" + 0.031*"treatment" + '
  '0.030*"let" + 0.028*"member" + 0.026*"pfizer" + 0.026*"healthy" + '
  '0.024*"next" + 0.023*"month"'),
 (1,
  '0.077*"pandemic" + 0.043*"help" + 0.041*"call" + 0.036*"drug" + '
  '0.028*"concern" + 0.028*"put" + 0.025*"ivermectin_multifacete" + '
  '0.023*"keep" + 0.019*"student" + 0.018*"online"'),
 (2,
  '0.072*"tell" + 0.053*"due" + 0.051*"use" + 0.049*"school" + 0.036*"start" + '
  '0.024*"public" + 0.023*"much" + 0.019*"ever" + 0.018*"post" + '
  '0.016*"available"'),
 (3,
  '0.078*"slots_age" + 0.074*"ages_date" + 0.074*"sep_fee" + 0.060*"time" + '
  '0.043*"hospital" + 0.028*"home" + 0.028*"covaxin_dose" + 0.027*"talk" + '
  '0.018*"delta_variant" + 0.015*"sound"'),
 (4,
  '0.281*"vaccine" + 0.051*"well" + 0.050*"mandatory" + 0.042*"give" + '
  '0.041*"world" + 0.028*"show" + 0.013*"fda_approval" + 0.012*"trend" + '
  '0.011*"severe" + 0.011*"prevent"'),
 (5,
  '0.177*"people" + 0.074*"go" + 0.04

In [ ]:
for idx, topic in lda_model.show_topics(formatted=False, num_topics=20, num_words= 100):
    print('Topic: {} \nWords: {}'.format(idx, '|'.join([w[0] for w in topic])))


Topic: 0 
Words: vaccinate|think|last|treatment|let|member|pfizer|healthy|next|month|infect|manage|away|hope|less|group|sign|result|smartnew|apparently|stay|immunity|qualifi|centre|imagine|suppose|fully|wcepi|cure|petition|covidiot|whole|speak|past|year_old|contract|percent|transmit|damage|tourist|concerned|notice|attack|holocaust|immediately|biden|come_later|natural|scary|antibody|similar|recognize|island|embrace|veteran|dose_regiman|ability|liberal|airline|woman_die|amount|trade|file|oppose|candidate|diagnosis|brain_disease|suddenly|firm|special|insider|sister|deltavariant|gold|twice|favorite|generate|achieve_herd|japanese|lagos_consider|drunk|section|goal|world_cup|visa|bet|drink|sebgorka|kidney|seven_minute|movie|speedy_recovery|hit_peak|snake|exam|epidemiology|running_ex|foundation|randomized_controlle|niece
Topic: 1 
Words: pandemic|help|call|drug|concern|put|ivermectin_multifacete|keep|student|online|rise|indicated_efficacy|check|least|understand|horrible|fight|chat|visit|friend

In [ ]:
from gensim.parsing.preprocessing import preprocess_string, strip_punctuation, strip_numeric

lda_topics = lda_model.show_topics(num_topics=20, num_words=50)

topics = []
filters = [lambda x: x.lower(), strip_punctuation, strip_numeric]

for topic in lda_topics:
    print(topic)
    topics.append(preprocess_string(topic[1], filters))

print(topics)

(0, '0.096*"vaccinate" + 0.061*"think" + 0.036*"last" + 0.031*"treatment" + 0.030*"let" + 0.028*"member" + 0.026*"pfizer" + 0.026*"healthy" + 0.024*"next" + 0.023*"month" + 0.020*"infect" + 0.019*"manage" + 0.018*"away" + 0.015*"hope" + 0.012*"less" + 0.012*"group" + 0.011*"sign" + 0.011*"result" + 0.011*"smartnew" + 0.011*"apparently" + 0.010*"stay" + 0.010*"immunity" + 0.009*"qualifi" + 0.008*"centre" + 0.007*"imagine" + 0.007*"suppose" + 0.007*"fully" + 0.007*"wcepi" + 0.006*"cure" + 0.006*"petition" + 0.006*"covidiot" + 0.006*"whole" + 0.005*"speak" + 0.005*"past" + 0.005*"year_old" + 0.005*"contract" + 0.005*"percent" + 0.004*"transmit" + 0.004*"damage" + 0.004*"tourist" + 0.004*"concerned" + 0.004*"notice" + 0.004*"attack" + 0.003*"holocaust" + 0.003*"immediately" + 0.003*"biden" + 0.003*"come_later" + 0.003*"natural" + 0.003*"scary" + 0.003*"antibody"')
(1, '0.077*"pandemic" + 0.043*"help" + 0.041*"call" + 0.036*"drug" + 0.028*"concern" + 0.028*"put" + 0.025*"ivermectin_multifac

In [ ]:
doc_lda = lda_model[corpus]

In [ ]:
# Compute Perplexity
print('\nPerplexity: ', lda_model.log_perplexity(corpus))  



Perplexity:  -9.655250938073756


In [ ]:
# Compute Coherence Score
from gensim.models import CoherenceModel# spaCy for preprocessing
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)


Coherence Score:  0.4135875416870716


In [ ]:
!pip3 install pyLDAvis==2.1.2

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 1.6 MB 5.1 MB/s 
  Created wheel for pyLDAvis: filename=pyLDAvis-2.1.2-py2.py3-none-any.whl size=97738 sha256=2de19d75742db41d2307090d4aebc5c67089673db9d6e44e295d2aa2602a54d2
  Stored in directory: /root/.cache/pip/wheels/3b/fb/41/e32e5312da9f440d34c4eff0d2207b46dc9332a7b931ef1e89
Successfully built pyLDAvis


In [ ]:
# Visualize the topics
import pyLDAvis.gensim
#import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
vis

/usr/local/lib/python3.7/dist-packages/past/types/oldstr.py:5: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  from collections import Iterable
/usr/local/lib/python3.7/dist-packages/pyLDAvis/_prepare.py:232: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  head(R).drop('saliency', 1)


PreparedData(topic_coordinates=                  x         y  topics  cluster       Freq
topic                                                    
14     4.364153e-01  0.105392       1        1  11.678513
9      8.623327e-02 -0.415687       2        1   9.081365
5     -1.649618e-02  0.016582       3        1   5.420322
15    -1.641694e-02  0.011326       4        1   5.084653
8     -2.453360e-02  0.015269       5        1   4.839262
18    -2.536145e-02  0.013487       6        1   4.810268
1     -1.890428e-02  0.015708       7        1   4.686949
2     -2.736303e-02  0.022490       8        1   4.614570
4     -1.981817e-02  0.005830       9        1   4.543533
6     -3.242218e-02  0.015625      10        1   4.522160
0     -2.637609e-02  0.013693      11        1   4.483026
17    -3.046747e-02  0.017012      12        1   4.469067
10    -3.275930e-02  0.012338      13        1   4.285808
11    -3.673847e-02  0.024555      14        1   4.099490
7     -4.288850e-02  0.022338      15        1   4.098063
3     -6.752562e-08  0.005404      16        1   4.084056
16    -3.983459e-02  0.014907      17        1   4.083997
13    -4.526361e-02  0.026271      18        1   3.915363
19    -3.720846e-02  0.027808      19        1   3.786720
12    -4.979619e-02  0.029653      20        1   3.412816, topic_info=             Term          Freq         Total Category  logprob  loglift
5           covid  25276.000000  25276.000000  Default  30.0000  30.0000
19        vaccine   5727.000000   5727.000000  Default  29.0000  29.0000
4            case   6392.000000   6392.000000  Default  28.0000  28.0000
60         people   4312.000000   4312.000000  Default  27.0000  27.0000
14            get   3788.000000   3788.000000  Default  26.0000  26.0000
...           ...           ...           ...      ...      ...      ...
1195     diagnose    106.264412    107.121846  Topic20  -4.9699   3.3696
2220      realize    105.684037    106.541470  Topic20  -4.9753   3.3696
3214           sc    102.236629    103.094063  Topic20  -5.0085   3.3693
51        failure     99.040163     99.897596  Topic20  -5.0403   3.3690
2638  daily_thank     98.808027     99.665460  Topic20  -5.0426   3.3690

[646 rows x 6 columns], token_table=      Topic      Freq            Term
term                                 
1883      9  0.990926            able
1905      9  0.991963          access
866      16  0.990371          accord
5243      6  0.992584          accuse
121       5  0.996476             act
...     ...       ...             ...
435       2  0.999069            year
76       19  0.995053       yesterday
439       7  0.992586             yet
785      19  0.998701           young
1519      6  0.991968  zealand_report

[720 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[15, 10, 6, 16, 9, 19, 2, 3, 5, 7, 1, 18, 11, 12, 8, 4, 17, 14, 20, 13])

In [ ]:
pyLDAvis.save_html(vis, '2021_lda.html')

In [ ]:
!unzip "/content/mallet-2.0.8.zip"

In [ ]:
mallet_path = '/content/mallet-2.0.8/bin/mallet' # update this path
ldamallet = gensim.models.wrappers.LdaMallet(mallet_path, 
                                             corpus=corpus, 
                                             num_topics=20,
                                             id2word=id2word,
                                             iterations=1000,
                                             optimize_interval=10)

/usr/local/lib/python3.7/dist-packages/smart_open/smart_open_lib.py:494: DeprecationWarning: This function is deprecated.  See https://github.com/RaRe-Technologies/smart_open/blob/develop/MIGRATING_FROM_OLDER_VERSIONS.rst for more information
  warnings.warn(message, category=DeprecationWarning)
/usr/local/lib/python3.7/dist-packages/smart_open/smart_open_lib.py:494: DeprecationWarning: This function is deprecated.  See https://github.com/RaRe-Technologies/smart_open/blob/develop/MIGRATING_FROM_OLDER_VERSIONS.rst for more information
  warnings.warn(message, category=DeprecationWarning)


In [ ]:
pprint(ldamallet.show_topics(num_topics=20,num_words=100,formatted=False))

# Compute Coherence Score
coherence_model_ldamallet = CoherenceModel(model=ldamallet, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_ldamallet = coherence_model_ldamallet.get_coherence()

print('\n Coherence Score: ', round(coherence_ldamallet, 2))

[(0,
  [('world', 0.042105263157894736),
   ('home', 0.0374384236453202),
   ('covid', 0.03634949442571947),
   ('pandemic', 0.022919367383977183),
   ('drug', 0.02229712211563391),
   ('free', 0.02079336271713767),
   ('lead', 0.01944516463572725),
   ('testing', 0.019134042001555614),
   ('turn', 0.018148820326678767),
   ('restriction', 0.017422867513611617),
   ('reason', 0.014570910033704951),
   ('booster_shot', 0.014311641171895255),
   ('provide', 0.014000518537723619),
   ('stay', 0.013844957220637801),
   ('base', 0.01322271195229453),
   ('event', 0.01151153746435053),
   ('site', 0.009904070521130412),
   ('nobel_prize', 0.008970702618615505),
   ('education', 0.008296603577910292),
   ('keep_tab', 0.007052113041223749),
   ('grow', 0.0070002592688618095),
   ('employees_worke', 0.0070002592688618095),
   ('small', 0.00694840549649987),
   ('part', 0.006844697951775992),
   ('choice', 0.006844697951775992),
   ('nation', 0.006844697951775992),
   ('recommend', 0.00674099040

In [ ]:
mallet_lda_model = gensim.models.wrappers.ldamallet.malletmodel2ldamodel(ldamallet)

In [ ]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(mallet_lda_model, corpus, id2word,sort_topics=False)

/usr/local/lib/python3.7/dist-packages/pyLDAvis/_prepare.py:232: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  head(R).drop('saliency', 1)


In [ ]:
print(vis.topic_order)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [ ]:
pyLDAvis.save_html(vis, '2021_lda_mallet.html')

In [ ]:
vis

PreparedData(topic_coordinates=                  x         y  topics  cluster      Freq
topic                                                   
0      9.297136e-07 -0.000331       1        1  4.995213
1      8.825777e-05  0.000287       2        1  5.003269
2      9.972338e-05 -0.000118       3        1  4.968373
3      6.051582e-05  0.000694       4        1  4.996484
4     -4.836025e-04 -0.000078       5        1  5.011082
5     -1.522392e-04 -0.000315       6        1  4.994903
6      1.886958e-04  0.000357       7        1  5.091743
7      2.331085e-04  0.000171       8        1  4.993255
8     -1.920329e-04 -0.000152       9        1  5.004622
9      1.244929e-04  0.000099      10        1  4.984925
10     2.745039e-04  0.000760      11        1  5.019357
11    -1.169057e-04 -0.000583      12        1  5.012972
12     4.320524e-06  0.000007      13        1  4.982893
13     1.881813e-04 -0.000135      14        1  5.004517
14     7.688951e-04 -0.000213      15        1  5.000383
15     1.045390e-04 -0.000302      16        1  5.006071
16    -4.319554e-04 -0.000526      17        1  4.986514
17    -2.678491e-04  0.000490      18        1  5.000304
18     5.483668e-04 -0.000352      19        1  4.975376
19    -1.039946e-03  0.000239      20        1  4.967742, topic_info=                 Term       Freq      Total Category  logprob  loglift
9740          takaful  15.000000  15.000000  Default  30.0000  30.0000
17790   brexitreality  15.000000  15.000000  Default  29.0000  29.0000
5164   northernbeache  15.000000  15.000000  Default  28.0000  28.0000
22845            redu  15.000000  15.000000  Default  27.0000  27.0000
19846             adj  16.000000  16.000000  Default  26.0000  26.0000
...               ...        ...        ...      ...      ...      ...
8481         dialogue   1.039384  16.526634  Topic20  -9.9726   0.2359
10298             spd   1.033981  16.100849  Topic20  -9.9778   0.2567
16474   misgovernance   1.031580  16.005861  Topic20  -9.9801   0.2603
27225          pourri   1.030358  15.989874  Topic20  -9.9813   0.2602
17392           ehite   1.030436  16.160740  Topic20  -9.9812   0.2496

[857 rows x 6 columns], token_table=       Topic      Freq     Term
term                           
16784      1  0.062294   abcent
16784      2  0.062294   abcent
16784      3  0.062294   abcent
16784      4  0.062294   abcent
16784      5  0.062294   abcent
...      ...       ...      ...
21101     16  0.062965  zwergie
21101     17  0.062965  zwergie
21101     18  0.062965  zwergie
21101     19  0.062965  zwergie
21101     20  0.062965  zwergie

[16679 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])